## 🇧🇷 Seja bem vindo!
Me chamo Ígor Silva, sou estudante de ciência de dados.

Neste estudo, realizamos um comparativo entre três métodos simples para vetorização de variáveis categóricas:
* OneHotEncoder
* Dummy Encoding
* Effect Encoding

O objetivo é entender o comportamento de cada método e em quais situações devemos aplicar cada um.

Dataset utilizado:
Toy Dataset (um pequeno subconjunto)

Para saber mais, acesse:
* [LinkedIn](https://www.linkedin.com/in/igordrsilva/)
* [Github](https://github.com/igordrsilva)

Divirta-se!

<br>

***

<br>
## 🇺🇸 Welcome!
My name is Igor Silva, and I am a Data Science student.

In this notebook, I compare three simple methods for encoding categorical variables:
* OneHotEncoder
* Dummy Encoding
* Effect Encoding

The goal is to understand how each method works and when to use each one.

Dataset used:
Toy dataset (a small sample)

To see more, visit:
* [LinkedIn](https://www.linkedin.com/in/igordrsilva/)
* [Github](https://github.com/igordrsilva)

Enjoy!

---
## Encoding Categorical Variables 🔢
---

This process is essential for feeding categorical data to machine learning models, like linear models and SVMs, that require numerical input.

## ⤵️ Imports

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
import pandas as pd

***

## Types of Encoding for Categorical Variables
Let's use the table above as a sample toy dataset

In [ ]:
table = [
    ["Sunny", "Hot", "High", "Weak", "No"],
    ["Sunny", "Hot", "High", "Strong", "No"],
    ["Overcast", "Hot", "High", "Weak", "Yes"],
    ["Rain", "Mild", "High", "Weak", "Yes"],
    ["Rain", "Cool", "Normal", "Weak", "Yes"],
    ["Rain", "Cool", "Normal", "Strong", "No"],
    ["Overcast", "Cool", "Normal", "Weak", "Yes"],
    ["Sunny", "Mild", "High", "Weak", "No"],
    ["Sunny", "Cool", "Normal", "Weak", "Yes"],
    ["Rain", "Mild", "Normal", "Strong", "Yes"],
    ["Sunny", "Mild", "Normal", "Strong", "Yes"],
    ["Overcast", "Mild", "High", "Strong", "Yes"],
    ["Overcast", "Hot", "Normal", "Weak", "Yes"],
    ["Rain", "Mild", "High", "Strong", "No"],
]

# Target variable: Play Tennis?
columns = [
    "outlook",
    "temperature",
    "humidity",
    "wind",
    "target"
]

In [ ]:
table = pd.DataFrame(table, columns=columns)

#### 📍 OneHotEncoder

The `OneHotEncoder`, a class from scikit-learn, is a utility that transforms categorical (non-numeric) data into a one-hot encoded numeric array.

It works by encoding each category value into a separate dimension (column) in the dataset. In this representation, the new column contains the value 1 when the observation belongs to that category and 0 otherwise.

<br>

| Advantages | Desadvantages |
|------------|---------------|
| Easy to interpret | Increases dimensionality |
| Compatible with many models | Not ideal for high-cardinality variables |
| Simple implementation | Higher memory usage |

<br>

See the implementation in the example below:

In [ ]:
encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

categorical_cols = [
  "outlook",
  "temperature",
  "humidity",
  "wind"
]

features_ohe = encoder.fit_transform(table[categorical_cols])

names_ohe = encoder.get_feature_names_out(categorical_cols)
table_ohe = pd.DataFrame(features_ohe, columns=names_ohe)

table_ohe.head()

,outlook_Overcast,outlook_Rain,outlook_Sunny,temperature_Cool,temperature_Hot,temperature_Mild,humidity_High,humidity_Normal,wind_Strong,wind_Weak
0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
4,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0


***

#### 📍 Dummy Encoding

`Dummy Encoding` is an alternative to One-Hot Encoding when you want to avoid linearly dependent features. It creates a vector of binary (0/1) indicators representing the categories of a categorical variable.

Unlike One-Hot Encoding, Dummy Encoding typically drops one of the generated columns. This prevents the dummy variable trap, a situation where one variable can be perfectly predicted from the others, leading to multicollinearity in some models.

In practice, if a categorical variable has k categories, Dummy Encoding generates k − 1 binary columns instead of k.

<br>

| Advantages | Desadvantages |
|------------|---------------|
| Avoids multicollinearity | Reference category information is implicit |
| Reduces number of columns | Still increases dimensionality |
| Works well with linear models | Harder to interpret for beginners |
| Efficient representation | Not suitable for high-cardinality variables|

<br>

See the implementation in the example below:

In [ ]:
table_dummies = pd.get_dummies(table[categorical_cols], columns=categorical_cols, drop_first=True)
table_dummies.head()

,outlook_Rain,outlook_Sunny,temperature_Hot,temperature_Mild,humidity_Normal,wind_Weak
0,False,True,True,False,False,True
1,False,True,True,False,False,False
2,False,False,True,False,False,True
3,True,False,False,True,False,True
4,True,False,False,False,True,True


***

#### 📍 Effect Encoding

`Effect Encoding` is a technique used to transform categorical variables into numerical features. It's similar to dummy encoding, but instead of using 0 as the baseline for the reference category, it represents the reference category using −1.

In this approach, if a categorical variable has k categories, the encoding generates k − 1 columns, and the remaining category is represented by −1 values across all encoded columns.

This method allows the model to interpret the encoded variables relative to the overall mean rather than a single reference category.
<br>

| Advantages | Desadvantages |
|------------|---------------|
| Useful for regression models | Harder to interpret |
| Avoids multicollinearity | Still increases dimensionality |
| Symmetric coefficient interpretation | Not ideal for tree models |

<br>

See the implementation in the example below:

In [ ]:
table_effect = table[categorical_cols].copy()

for col in categorical_cols:
    unique_categories = table[col].unique()
    reference_category = unique_categories[0]

    for i, category in enumerate(unique_categories):
        if category == reference_category:
            continue

        new_col_name = f'{col}_{category}'
        table_effect[new_col_name] = 0
        table_effect.loc[table_effect[col] == category, new_col_name] = 1
        table_effect.loc[table_effect[col] == reference_category, new_col_name] = -1

    table_effect = table_effect.drop(columns=[col])


table_effect.head()

,outlook_Overcast,outlook_Rain,temperature_Mild,temperature_Cool,humidity_Normal,wind_Strong
0,-1,-1,-1,-1,-1,-1
1,-1,-1,-1,-1,-1,1
2,1,0,-1,-1,-1,-1
3,0,1,1,0,-1,-1
4,0,1,0,1,1,-1


***